# Lab 3. Spark structured Streaming - okna na żywym strumieniu

In [12]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("okienka").getOrCreate()
spark

In [13]:
czujnik_temperatury = ((12.5, "2019-01-02 12:00:00"),
(17.6, "2019-01-02 12:00:20"),
(14.6,  "2019-01-02 12:00:30"),
(22.9,  "2019-01-02 12:01:15"),
(17.4,  "2019-01-02 12:01:30"),
(25.8,  "2019-01-02 12:03:25"),
(27.1,  "2019-01-02 12:02:40"),
)

In [21]:
from pyspark.sql.functions import to_timestamp
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

schema = StructType([
    StructField("temperatura", DoubleType(), True),
    StructField("czas", StringType(), True),
])

SCHEMA = """
temperatura DOUBLE, czas STRING
""" # schemat DDL # nazwa typ, nazwa typ, jako długi string

In [22]:
df = (spark.createDataFrame(czujnik_temperatury, schema=schema)
      .withColumn("czas", to_timestamp("czas")))

df2 = (spark.createDataFrame(czujnik_temperatury, schema=SCHEMA)
      .withColumn("czas", to_timestamp("czas")))

In [23]:
df.printSchema()

root
 |-- temperatura: double (nullable = true)
 |-- czas: timestamp (nullable = true)



In [24]:
df2.printSchema()

root
 |-- temperatura: double (nullable = true)
 |-- czas: timestamp (nullable = true)



In [25]:
df.show(2)

+-----------+-------------------+
|temperatura|               czas|
+-----------+-------------------+
|       12.5|2019-01-02 12:00:00|
|       17.6|2019-01-02 12:00:20|
+-----------+-------------------+
only showing top 2 rows



In [20]:
df2.show(2)

+-----------+-------------------+
|temperature|               czas|
+-----------+-------------------+
|       12.5|2019-01-02 12:00:00|
|       17.6|2019-01-02 12:00:20|
+-----------+-------------------+
only showing top 2 rows



Uzyskaliśmy 2 takie same tabelki

## Przechodzimy do trybu SQL-owego

In [27]:
df.createOrReplaceTempView("df") # Widok tymczasowy

In [30]:
wynik = spark.sql("select czas, temperatura from df where temperatura > 21").show(5)
wynik

+-------------------+-----------+
|               czas|temperatura|
+-------------------+-----------+
|2019-01-02 12:01:15|       22.9|
|2019-01-02 12:03:25|       25.8|
|2019-01-02 12:02:40|       27.1|
+-------------------+-----------+



In [31]:
# Thumbling window

import pyspark.sql.functions as F

df3 = df.groupBy(F.window("czas","30 seconds")).count()
df3.show(truncate=False)

+------------------------------------------+-----+
|window                                    |count|
+------------------------------------------+-----+
|{2019-01-02 12:00:00, 2019-01-02 12:00:30}|2    |
|{2019-01-02 12:00:30, 2019-01-02 12:01:00}|1    |
|{2019-01-02 12:01:00, 2019-01-02 12:01:30}|1    |
|{2019-01-02 12:01:30, 2019-01-02 12:02:00}|1    |
|{2019-01-02 12:03:00, 2019-01-02 12:03:30}|1    |
|{2019-01-02 12:02:30, 2019-01-02 12:03:00}|1    |
+------------------------------------------+-----+



In [32]:
df3.printSchema()

root
 |-- window: struct (nullable = false)
 |    |-- start: timestamp (nullable = true)
 |    |-- end: timestamp (nullable = true)
 |-- count: long (nullable = false)



In [33]:
spark.stop()

## Strumieniowanie

In [ ]:
df = spark.readStream.format("rate").option("rowsPerSecond", 1).load()

- co sekunde generowanie danych
- 1 row na sekunde
- 1 kolumna: znacznik czasu
- 2 kolumna: value (licznik rosnący)

In [36]:
%%file streamrate.py
## uruchom przez spark-submit streamrate.py
## spakr-submit - skypty Pythonowe w Sparku
## Skrypt przetwarzania strumieniowego


from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("StreamingDemo").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

df = (spark.readStream
      .format("rate")
      .option("rowsPerSecond", 1)
      .load()
)

# Ukryta transofrmacja df = 1*df
query = (df.writeStream 
    .format("console")          # Efekty w terminalu
    .outputMode("append")       # Domyślnie też na append
    .option("truncate", False)  # Wyświetlanie w całości
    .start()
) 

query.awaitTermination()    # Jeżeli kod jako sktrypt to ten kod zapewnia poprawne zakończenie działania

Writing streamrate.py


In [37]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("StreamingDemo").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

# Jeżeli nie skrypt tylko w notatniki to muismy dodać funkcję:
def process_batch(df, batch_id, tstop=5):
    print(f"Batch ID: {batch_id}")
    df.show(truncate=False)
    if batch_id == tstop:
        df.stop()


# Załadowanie identyczne
df = (spark.readStream
      .format("rate")
      .option("rowsPerSecond", 1)
      .load()
)

# Prawie identyczne
query = (df.writeStream 
    .format("console") 
    .outputMode("append")
    .foreachBatch(process_batch)  # Dla każdego batcha wykonaj process_batch
    .option("truncate", False) 
    .start()
)

Batch ID: 0
+---------+-----+
|timestamp|value|
+---------+-----+
+---------+-----+

Batch ID: 1
+-----------------------+-----+
|timestamp              |value|
+-----------------------+-----+
|2026-04-20 10:27:42.181|0    |
+-----------------------+-----+

Batch ID: 2
+-----------------------+-----+
|timestamp              |value|
+-----------------------+-----+
|2026-04-20 10:27:43.181|1    |
+-----------------------+-----+

Batch ID: 3
+-----------------------+-----+
|timestamp              |value|
+-----------------------+-----+
|2026-04-20 10:27:44.181|2    |
+-----------------------+-----+

Batch ID: 4
+-----------------------+-----+
|timestamp              |value|
+-----------------------+-----+
|2026-04-20 10:27:45.181|3    |
+-----------------------+-----+

Batch ID: 5
+-----------------------+-----+
|timestamp              |value|
+-----------------------+-----+
|2026-04-20 10:27:46.181|4    |
+-----------------------+-----+



In [ ]:
spakr.stop() # Lub restart kernela

In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("StreamingDemo").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

def process_batch(df, batch_id, tstop=10):
    print(f"Batch ID: {batch_id}")
    df.show(truncate=False)
    if batch_id == tstop:
        df.stop()

from pyspark.sql.functions import col, expr

df = (spark.readStream
      .format("rate")
      .option("rowsPerSecond", 1)
      .load()
)

stream = (df.withColumn("czas", col("timestamp"))
        .withColumn("temperatura", expr("20 + rand() * 10"))
        .select("czas", "temperatura")
       )

query = (stream.writeStream 
    .format("console") 
    .outputMode("append")
    .foreachBatch(process_batch)
    .option("truncate", False) 
    .start()
)

Batch ID: 0
+----+-----------+
|czas|temperatura|
+----+-----------+
+----+-----------+

Batch ID: 1
+----------------------+------------------+
|czas                  |temperatura       |
+----------------------+------------------+
|2026-04-20 10:32:26.91|26.692919186985975|
+----------------------+------------------+

Batch ID: 2
+----------------------+-----------------+
|czas                  |temperatura      |
+----------------------+-----------------+
|2026-04-20 10:32:27.91|25.24253264521363|
+----------------------+-----------------+

Batch ID: 3
+----------------------+---------------+
|czas                  |temperatura    |
+----------------------+---------------+
|2026-04-20 10:32:28.91|21.503794620613|
+----------------------+---------------+

Batch ID: 4
+----------------------+------------------+
|czas                  |temperatura       |
+----------------------+------------------+
|2026-04-20 10:32:29.91|27.901162888285143|
+----------------------+------------------+


Restart kernela

## 1. Transformacja bezstanowa

In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("StreamingDemo").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

def process_batch(df, batch_id, tstop=5):
    print(f"Batch ID: {batch_id}")
    df.show(truncate=False)
    if batch_id == tstop:
        df.stop()

from pyspark.sql.functions import col, expr

df = (spark.readStream
      .format("rate")
      .option("rowsPerSecond", 1)
      .load()
)

stream = (df.withColumn("czas", col("timestamp"))
        .withColumn("temperatura", expr("20 + rand() * 10"))
        .select("czas", "temperatura")
       )

stream_filtered = stream.filter(col("temperatura") > 25)

query = (stream_filtered.writeStream 
    .format("console") 
    .outputMode("append")
    .foreachBatch(process_batch)
    .option("truncate", False) 
    .start()
)

Batch ID: 0
+----+-----------+
|czas|temperatura|
+----+-----------+
+----+-----------+

Batch ID: 1
+----+-----------+
|czas|temperatura|
+----+-----------+
+----+-----------+

Batch ID: 2
+----+-----------+
|czas|temperatura|
+----+-----------+
+----+-----------+

Batch ID: 3
+-----------------------+------------------+
|czas                   |temperatura       |
+-----------------------+------------------+
|2026-04-20 10:42:41.634|29.917857007862143|
+-----------------------+------------------+

Batch ID: 4
+-----------------------+------------------+
|czas                   |temperatura       |
+-----------------------+------------------+
|2026-04-20 10:42:42.634|25.842580911460328|
+-----------------------+------------------+

Batch ID: 5
+----+-----------+
|czas|temperatura|
+----+-----------+
+----+-----------+



Restart Kernela

In [ ]:
## 2. Prosty model wykrywający anomialie

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, expr, when

spark = SparkSession.builder.appName("StreamingDemo").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

def process_batch(df, batch_id, tstop=10):
    print(f"Batch ID: {batch_id}")
    df.show(truncate=False)
    if batch_id == tstop:
        df.stop()



df = (spark.readStream
      .format("rate")
      .option("rowsPerSecond", 1)
      .load()
)

stream = (df.withColumn("czas", col("timestamp"))
        .withColumn("temperatura", expr("20 + rand() * 10"))
        .select("czas", "temperatura")
       )

stream_anomaly = stream.withColumn(
    "anomaly",
    when(col("temperatura") > 29.5, "TAK").otherwise("NIE")
)

query = (stream_anomaly.writeStream 
    .format("console") 
    .outputMode("append")
    .foreachBatch(process_batch)
    .option("truncate", False) 
    .start()
)

Batch ID: 0
+----+-----------+-------+
|czas|temperatura|anomaly|
+----+-----------+-------+
+----+-----------+-------+

Batch ID: 1
+-----------------------+------------------+-------+
|czas                   |temperatura       |anomaly|
+-----------------------+------------------+-------+
|2026-04-20 10:45:18.345|21.571128134891616|NIE    |
+-----------------------+------------------+-------+

Batch ID: 2
+-----------------------+------------------+-------+
|czas                   |temperatura       |anomaly|
+-----------------------+------------------+-------+
|2026-04-20 10:45:19.345|29.319771375174586|NIE    |
+-----------------------+------------------+-------+

Batch ID: 3
+-----------------------+-----------------+-------+
|czas                   |temperatura      |anomaly|
+-----------------------+-----------------+-------+
|2026-04-20 10:45:20.345|24.07200526487523|NIE    |
+-----------------------+-----------------+-------+

Batch ID: 4
+-----------------------+-------------

Pierwsza ramka jest zazwyczaj pusta

Restart kernela

## 3. Tumbling widnow

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import window, col, expr

spark = SparkSession.builder.appName("StreamingDemo").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

def process_batch(df, batch_id, tstop=50):
    print(f"Batch ID: {batch_id}")
    df.show(truncate=False)
    if batch_id == tstop:
        df.stop()



df = (spark.readStream
      .format("rate")
      .option("rowsPerSecond", 1)
      .load()
)

stream = (df.withColumn("czas", col("timestamp"))
        .withColumn("temperatura", expr("20 + rand() * 10"))
        .select("czas", "temperatura")
       )

# Konwertuj 'temperatura' na typ Double, aby agregacja była możliwa
stream = stream.withColumn("temperatura", col("temperatura").cast("double"))

tumbling_window = (stream
    #.groupBy(window(col("czas"), "10 seconds"))   # Najpierw grupuj po oknie, a pozniej licz srednią
    .groupBy(window(col("czas"), "10 seconds", "5 seconds"))    # Okno przesuwne
    .avg("temperatura").alias('srednia')           # Przez aggregacje spark nie pozwoli korzystać z append
    .orderBy(col('window.start'))
)

query = (tumbling_window.writeStream 
    .format("console") 
    .outputMode("complete")               # Za każdym razem dostajemy całą ramke, a nie tylko nowe srednie
    .foreachBatch(process_batch)
    .option("truncate", False) 
    .start()
)

Batch ID: 0
+------+----------------+
|window|avg(temperatura)|
+------+----------------+
+------+----------------+

Batch ID: 1
+------------------------------------------+------------------+
|window                                    |avg(temperatura)  |
+------------------------------------------+------------------+
|{2026-04-20 10:58:20, 2026-04-20 10:58:30}|21.51091375008605 |
|{2026-04-20 10:58:25, 2026-04-20 10:58:35}|23.841012880817303|
|{2026-04-20 10:58:30, 2026-04-20 10:58:40}|25.23907235925605 |
+------------------------------------------+------------------+

Batch ID: 2
+------------------------------------------+------------------+
|window                                    |avg(temperatura)  |
+------------------------------------------+------------------+
|{2026-04-20 10:58:20, 2026-04-20 10:58:30}|21.51091375008605 |
|{2026-04-20 10:58:25, 2026-04-20 10:58:35}|23.841012880817303|
|{2026-04-20 10:58:30, 2026-04-20 10:58:40}|25.717133775428657|
|{2026-04-20 10:58:35, 202

## Generator danych

In [1]:
%%file generator.py
# generator.py
import json, os, random, time
from datetime import datetime, timedelta

output_dir = "data/stream"
os.makedirs(output_dir, exist_ok=True)

sklepy = ['Warszawa', 'Kraków', 'Gdańsk', 'Wrocław']
kategorie = ['elektronika', 'odzież', 'żywność', 'książki']

def generate_transaction():
    return {
        'tx_id': f'TX{random.randint(1000,9999)}',
        'user_id': f'u{random.randint(1,20):02d}',
        'amount': round(random.uniform(5.0, 5000.0), 2),
        'store': random.choice(sklepy),
        'category': random.choice(kategorie),
        'timestamp': datetime.now().isoformat(),
    }

# Simulate file-based streaming
while True:
    batch = [generate_transaction() for _ in range(2)]
    filename = f"{output_dir}/events_{int(time.time())}.json"
    with open(filename, "w") as f:
        for e in batch:
            f.write(json.dumps(e) + "\n")
    print(f"Wrote: {filename}")
    time.sleep(5)

Writing generator.py


In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
from pyspark.sql.functions import from_json, to_timestamp
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("jsonDemo").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

tx_schema = StructType([
    StructField("tx_id",     StringType()),
    StructField("user_id",   StringType()),
    StructField("amount",    DoubleType()),
    StructField("store",     StringType()),
    StructField("category",  StringType()),
    StructField("timestamp", StringType()),
])

batch_counter = {"count": 0}

def process_batch(df, batch_id, tstop=20):
    batch_counter["count"] += 1
    print(f"Batch ID: {batch_id}")
    df.show(truncate=False)
    if batch_id == tstop:
        df.stop()
    


stream = (spark.readStream
          .schema(tx_schema)
          .json("data/stream"))

query = (stream.writeStream
         .format("console")
         .foreachBatch(process_batch)
         .start())

Pobieramy jsony z katalogu, które co 5 sekund powinny być generowane przez generator.
Jeżeli generator.py działą to dopisują sie następne jsony (append)

Spróbować stworzyć filtrowania albo agregację (i wyłączyć generator).